# NumCompute Quickstart Guide

**NumCompute**, a modular, end-to-end machine learning framework built from scratch using only Python and NumPy. This notebook demonstrates the core features of the library, including data handling, preprocessing, pipelines, statistical utilities, and optimization.

## 1. Setup and Imports

First, we'll import the necessary modules from `numcompute`.

In [25]:
import numpy as np
import os

# Core Modules
from numcompute.io import load_csv
from numcompute.preprocessing import StandardScaler, MinMaxScaler, SimpleImputer, OneHotEncoder
from numcompute.pipeline import Pipeline
from numcompute.stats import mean, variance, welford, histogram, quantile
from numcompute.metrics import accuracy, confusion_matrix, f1, mse, roc_curve, auc
from numcompute.sort_search import topk, binary_search, multi_key_sort
from numcompute.optim import grad, jacobian, line_search
from numcompute.benchmarking import run_all_benchmarks

print("NumCompute modules imported successfully!")

NumCompute modules imported successfully!


## 2. Data Loading

We use `load_csv` to load datasets. It handles headers and missing values.

In [26]:
# Load the Iris dataset
data = load_csv('iris.csv', skip_rows=1)
print(f"Dataset shape: {data.shape}")
print(f"First 5 rows:\n{data[:5]}")

# Load dataset with missing values
data_missing = load_csv('iris_missing.csv', skip_rows=1)
nan_count = np.isnan(data_missing).sum()
print(f"\nMissing values in iris_missing.csv: {nan_count}")

Dataset shape: (150, 5)
First 5 rows:
[[5.1 3.5 1.4 0.2 nan]
 [4.9 3.  1.4 0.2 nan]
 [4.7 3.2 1.3 0.2 nan]
 [4.6 3.1 1.5 0.2 nan]
 [5.  3.6 1.4 0.2 nan]]

Missing values in iris_missing.csv: 222


## 3. Preprocessing and Pipelines

NumCompute provides transformers for handling missing data and scaling features, which can be chained using a `Pipeline`.

In [27]:
# Extract features (first 4 columns)
X = data_missing[:, :4]

# Define a pipeline: Impute -> Scale
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Fit and transform the data
X_processed = pipe.fit_transform(X)

print(f"Processed data shape: {X_processed.shape}")
print(f"Mean after scaling: {np.round(np.mean(X_processed, axis=0), 4)}")
print(f"Std after scaling: {np.round(np.std(X_processed, axis=0), 4)}")

# One-Hot Encoding Example
labels = np.array(['setosa', 'versicolor', 'virginica', 'setosa'])
encoder = OneHotEncoder()
one_hot = encoder.fit_transform(labels)
print(f"\nCategories: {encoder.categories_}")
print(f"One-hot encoded:\n{one_hot}")

Processed data shape: (150, 4)
Mean after scaling: [-0. -0. -0. -0.]
Std after scaling: [1. 1. 1. 1.]

Categories: ['setosa' 'versicolor' 'virginica']
One-hot encoded:
[[1 0 0]
 [0 1 0]
 [0 0 1]
 [1 0 0]]


## 4. Statistical Utilities

Robust statistics with NaN handling and streaming algorithms.

In [28]:
x = data_missing[:, 0]  # First column with NaNs

print(f"Mean (ignoring NaNs): {mean(x, ignore_nan=True):.4f}")
print(f"Variance (ignoring NaNs): {variance(x, ignore_nan=True):.4f}")

# Welford's algorithm for streaming mean and variance
w_mean, w_var = welford(x)
print(f"Welford Mean: {w_mean:.4f}, Welford Var: {w_var:.4f}")

# Quantiles
q = quantile(x[~np.isnan(x)], [0.25, 0.5, 0.75])
print(f"25th, 50th, 75th percentiles: {q}")

Mean (ignoring NaNs): 5.8432
Variance (ignoring NaNs): 0.6825
Welford Mean: 5.8432, Welford Var: 0.6877
25th, 50th, 75th percentiles: [5.1  5.75 6.4 ]


## 5. Metrics

Evaluate your models with classification and regression metrics.

In [29]:
y_true = np.array([0, 1, 1, 0, 1])
y_pred = np.array([0, 1, 0, 0, 1])

print(f"Accuracy: {accuracy(y_true, y_pred)}")
print(f"F1 Score: {f1(y_true, y_pred):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}")

# ROC and AUC
y_scores = np.array([0.1, 0.4, 0.35, 0.8])
y_true_binary = np.array([0, 0, 1, 1])
fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
print(f"\nAUC: {auc(fpr, tpr):.4f}")

Accuracy: 0.8
F1 Score: 0.8000
Confusion Matrix:
[[2 0]
 [1 2]]

AUC: 0.7500


## 6. Optimization

Numerical optimization tools including gradient and Jacobian estimation.

In [30]:
# Scalar function: f(x) = x0^2 + x1^2
def f(x):
    return np.sum(x**2)

x0 = np.array([1.0, 2.0])
print(f"Gradient at {x0}: {grad(f, x0)}")

# Vector function: F(x) = [x0^2, x1^2]
def F(x):
    return x**2

print(f"Jacobian at {x0}:\n{jacobian(F, x0)}")

# Line search
direction = -grad(f, x0)
alpha = line_search(f, x0, direction)
print(f"Step size alpha: {alpha}")

Gradient at [1. 2.]: [2. 4.]
Jacobian at [1. 2.]:
[[2. 0.]
 [0. 4.]]
Step size alpha: 0.5


## 7. Benchmarking

Compare the performance of vectorised NumPy operations against pure Python loops.

In [31]:
run_all_benchmarks(n=100000)  # Reduced size for quick demo


Environment:
Array size: 100000
Top-k: 10
Seed: 42

Mean Benchmark
----------------------------------------
Function              Time (ms)
----------------------------------------
Mean (NumPy)             0.0485
Mean (Loop)              5.3438
----------------------------------------

Top-k Benchmark
----------------------------------------
Function              Time (ms)
----------------------------------------
Top-k (NumPy)            0.7415
Top-k (Loop)            16.8565
----------------------------------------
